# Laboratorium 1 - analiza koszykowa

## Przygotowanie

 * pobierz i wypakuj dataset: https://kaggle.com/datasets/rashikrahmanpritom/groceries-dataset-for-market-basket-analysismba?resource=download&select=basket.csv
   * alternatywnie, pobierz plik `basket.csv` z Teamsów
 * [opcjonalnie] Utwórz wirtualne środowisko
 `python3 -m venv ./recsyslab1`
 * zainstaluj potrzebne biblioteki:
 `pip install more-itertools`

## Część 1. - przygotowanie danych

In [1]:
# importujemy wszystkie potrzebne pakiety

import random
from typing import NamedTuple
from more_itertools import powerset

In [2]:
# definiujemy stałe

PATH = './basket.csv'
EPSILON = 0.0005
N = 20          # liczba rekomendacji zwracanych przez recommend()
K = [3, 5, 10]  # wartości k używane przy ewaluacji
N_EVAL = 200    # liczba koszyków użytych do ewaluacji
SEED = 42
TEST_FRACTION = 0.2

In [3]:
# wczytujemy dane o koszykach

def read_baskets(path: str) -> list[set[str]]:
    with open(path, encoding="utf-8") as f:
        raw = f.read()
    baskets = [
        {item.lower() for item in row.split(',') if item}
        for row in raw.split('\n')[1:]
        if row
    ]
    return baskets


def unique_products(baskets: list[set[str]]) -> list[str]:
    products: set[str] = set()
    for basket in baskets:
        products.update(basket)
    return sorted(products)


baskets = read_baskets(PATH)
products = unique_products(baskets)

split_rng = random.Random(SEED)

shuffled = baskets[:]
split_rng.shuffle(shuffled)
n_test = int(len(shuffled) * TEST_FRACTION)

train_baskets = shuffled[n_test:]
test_baskets = shuffled[:n_test]

print(f"Wczytano {len(baskets)} koszyków i {len(products)} unikalnych produktów.")
print(f"Treningowych: {len(train_baskets)}, testowych: {len(test_baskets)}")

Wczytano 14963 koszyków i 167 unikalnych produktów.
Treningowych: 11971, testowych: 2992


## Część 2. - obliczanie wskaźników

In [4]:
# obliczamy strukturę danych (słownik) przechowującą wszystkie interesujące wartości `support`

Supports = dict[str, float]


def key(products) -> str:
    return ','.join(sorted(products))


def get_supports(
    baskets: list[set[str]],
    epsilon: float,
) -> Supports:
    counts: dict[str, int] = {}
    for basket in baskets:
        for subset in powerset(basket):
            if subset:
                k = key(subset)
                counts[k] = counts.get(k, 0) + 1
    n = len(baskets)
    return {k: c / n for k, c in counts.items() if c / n >= epsilon}


supports = get_supports(train_baskets, EPSILON)
supports

{'curd': 0.034500041767605044,
 'domestic eggs': 0.03767437975106507,
 'cream cheese ': 0.02372399966585916,
 'curd,domestic eggs': 0.0010024225210926406,
 'cream cheese ,curd': 0.0008353521009105338,
 'napkins': 0.021134408153036504,
 'other vegetables': 0.12196140673293793,
 'napkins,other vegetables': 0.002171915462367388,
 'whipped/sour cream': 0.043855985297803025,
 'oil': 0.01553754907693593,
 'oil,whipped/sour cream': 0.0007518168908194805,
 'soda': 0.09740205496616824,
 'bottled water': 0.06089716815637791,
 'bottled water,soda': 0.004845042185281096,
 'tropical fruit': 0.06641049202238744,
 'ham': 0.01779299974939437,
 'ham,tropical fruit': 0.0011694929412747472,
 'turkey': 0.00593099991646479,
 'photo/film': 0.005095647815554256,
 'mayonnaise': 0.005012112605463203,
 'pip fruit': 0.049035168323448335,
 'dishes': 0.009105337899924818,
 'sliced cheese': 0.014451591345752234,
 'pip fruit,sliced cheese': 0.0005012112605463203,
 'coffee': 0.031492774204327126,
 'frankfurter': 0.03

In [5]:
# definiujemy funkcje obliczające support, confidence i lift

def support(supports: Supports, products: set[str]) -> float:
    if not products:
        return 1.0
    return supports.get(key(products), 0.0)


def confidence(
    supports: Supports,
    set_a: set[str],
    set_b: set[str],
) -> float:
    s_set_a = support(supports, set_a)
    if s_set_a == 0:
        return 0.0
    return support(supports, set_a | set_b) / s_set_a


def lift(
    supports: Supports,
    set_a: set[str],
    set_b: set[str],
) -> float:
    s_set_b = support(supports, set_b)
    if s_set_b == 0:
        return 0.0
    return confidence(supports, set_a, set_b) / s_set_b

In [6]:
print(support(supports, {'whole milk', 'rolls/buns'}))
print(confidence(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))
print(lift(supports, {'whole milk', 'rolls/buns'}, {'yogurt'}))

0.01386684487511486
0.10240963855421686
1.1856342196639558


## Część 3. - generowanie rekomendacji

In [7]:
# wyznaczamy listę potencjalnych rekomendacji
# rekomendowane artykuły powinny mieć lift > 1 i jak najwyższe confidence

class Candidate(NamedTuple):
    item: str
    subbasket: set[str]
    confidence: float
    lift: float


def generate_next_product_candidates(
    basket: set[str],
    products: list[str],
    supports: Supports,
) -> list[Candidate]:
    """Zwraca listę kandydatów posortowanych malejąco po confidence."""
    candidates = []
    for subset in powerset(basket):
        subbasket = set(subset)
        if not subbasket or support(supports, subbasket) == 0:
            continue
        for item in products:
            if item in basket:
                continue
            l = lift(supports, subbasket, {item})
            if l > 1:
                c = confidence(supports, subbasket, {item})
                candidates.append(Candidate(item, subbasket, c, l))
    candidates.sort(key=lambda c: (c.confidence, c.lift), reverse=True)
    return candidates

In [8]:
# wybieramy top-n unikalnych produktów z listy kandydatów

def recommend(
    basket: set[str],
    products: list[str],
    supports: Supports,
    n: int,
) -> list[str]:
    """Zwraca top-n rekomendacji na podstawie analizy koszykowej."""
    result: list[str] = []
    for candidate in generate_next_product_candidates(basket, products, supports):
        if candidate.item not in result:
            result.append(candidate.item)
        if len(result) == n:
            break
    return result

In [9]:
print("Koszyk:", baskets[1])
print("Rekomendacje:", recommend(baskets[1], products, supports, N))

Koszyk: {'yogurt', 'semi-finished bread', 'whole milk', 'sausage'}
Rekomendacje: ['other vegetables', 'rolls/buns', 'soda', 'pastry', 'tropical fruit', 'pork', 'margarine', 'shopping bags', 'root vegetables', 'bottled beer', 'citrus fruit', 'frozen vegetables', 'domestic eggs', 'onions', 'newspapers', 'canned beer', 'pip fruit', 'curd', 'frankfurter', 'butter']


In [10]:
print("Koszyk:", baskets[33])
print("Rekomendacje:", recommend(baskets[33], products, supports, N))

Koszyk: {'tropical fruit', 'white wine', 'yogurt', 'photo/film', 'soda', 'domestic eggs', 'root vegetables'}
Rekomendacje: ['whole milk', 'sausage', 'citrus fruit', 'pastry', 'pip fruit', 'beef', 'shopping bags', 'coffee', 'newspapers', 'bottled beer', 'brown bread', 'pork', 'frozen vegetables', 'chicken', 'waffles', 'white bread', 'cream cheese ', 'berries', 'hamburger meat', 'uht-milk']


## Część 4. - ewaluacja rekomendacji: MRR@k

### Metryka

**MRR@k** (Mean Reciprocal Rank) mierzy, na jakiej *pozycji* pojawił się ukryty produkt wśród top-k rekomendacji. Nagradza wyższe miejsca na liście:

$$\text{MRR@k} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\text{rank}_q}$$

gdzie:
- $\mathcal{Q}$ to zbiór przypadków testowych
- $\text{rank}_q$ to pozycja pierwszego trafnego produktu w top-k dla danego przypadku testowego $q$ (lub $\infty$, jeśli nie znaleziono — wkład do sumy wynosi wtedy 0).

### Protokół ewaluacji

Dla każdego koszyka testowego:
1. losowo wydzielamy jeden ukryty produkt; ten produkt byłby tym, który użytkownik najchętniej umieści w koszyku i tym samym tym, który powinien zaproponować nasz algorytm,
2. pozostałe produkty stanowią kontekst (czyli koszyk) przekazywany rekomendatorowi,
3. sprawdzamy, na której pozycji top-k pojawia się ukryty produkt.

Porównamy dwa algorytmy:
- **basket** – rekomendator oparty na analizie koszykowej (confidence + lift),
- **random** – rekomendator losowy (baseline).

Porównanie możemy przeprowadzić dla różnych wartości $k$.

In [11]:
# algorytm losowy - punkt odniesienia (baseline)

def recommend_random(
    basket: set[str],
    products: list[str],
    k: int,
    rng: random.Random | None = None,
) -> list[str]:
    """Zwraca k losowo wybranych produktów spoza koszyka."""
    rng = rng or random.Random()
    available = [p for p in products if p not in basket]
    return rng.sample(available, min(k, len(available)))

In [12]:
# implementacja metryki MRR@k

def reciprocal_rank_at_k(recommended: list[str], relevant: set[str], k: int) -> float:
    """Zwraca odwrotność pozycji pierwszego trafnego produktu w top-k (0 jeśli brak trafienia)."""
    for rank, item in enumerate(recommended[:k], start=1):
        if item in relevant:
            return 1 / rank
    return 0.0

In [13]:
# podział koszyka na kontekst i ukryte produkty (ground truth)

def split_basket(
    basket: set[str],
    sample_size: int,
    rng: random.Random,
) -> tuple[set[str], set[str]]:
    """Losowo wydziela `sample_size` produktów jako cel dla rekomendacji; reszta to koszyk"""
    hidden = set(rng.sample(sorted(basket), sample_size))
    return basket - hidden, hidden

In [14]:
# typy reprezentujące wyniki ewaluacji

class MetricAtK(NamedTuple):
    k: int
    mrr: float


class EvaluationResults(NamedTuple):
    basket: MetricAtK
    random: MetricAtK

In [15]:
# ewaluacja obu rekomendatorów dla zadanego k

def evaluate(
    baskets: list[set[str]],
    products: list[str],
    supports: Supports,
    k: int,
    n_eval: int = N_EVAL,
    seed: int = SEED,
) -> EvaluationResults:
    """
    Generuje `n_eval` przypadków testowych przy uzyciu funkcji `split_basket`.
    Dla kazdego przypadku testowego uruchamia oba algorytmy, a następnie oblicza `MMR@k`.
    Zwraca średnie wartości MMR@k dla wskazanego `k` i dla obu algorytmów.
    """
    rng = random.Random(seed)
    eligible = [b for b in baskets if len(b) >= 2]
    test_baskets = rng.sample(eligible, min(n_eval, len(eligible)))

    rr_basket = 0.0
    rr_random = 0.0
    for b in test_baskets:
        context, hidden = split_basket(b, 1, rng)
        rr_basket += reciprocal_rank_at_k(recommend(context, products, supports, k), hidden, k)
        rr_random += reciprocal_rank_at_k(recommend_random(context, products, k, rng), hidden, k)

    n = len(test_baskets)
    return EvaluationResults(
        basket=MetricAtK(k, rr_basket / n),
        random=MetricAtK(k, rr_random / n),
    )

In [16]:
# wyświetlamy wyniki i porównujemy algorytmy dla każdego k z listy K

def print_evaluation(results: EvaluationResults) -> None:
    col_w = 10
    header = f"{'Algorytm':<12} | {'mrr@' + str(results.basket.k):>{col_w}}"
    print(header)
    print('-' * len(header))
    for algo, metric in results._asdict().items():
        print(f"{algo:<12} | {metric.mrr:>{col_w}.4f}")


for k in K:
    print_evaluation(evaluate(test_baskets, products, supports, k))
    print()

Algorytm     |      mrr@3
-------------------------
basket       |     0.0550
random       |     0.0042

Algorytm     |      mrr@5
-------------------------
basket       |     0.0508
random       |     0.0167

Algorytm     |     mrr@10
-------------------------
basket       |     0.0423
random       |     0.0179



In [17]:
# porównanie wartości epsilon

for eps in [0.001, 0.0005, 0.0001, 0.00005, 0]:
    s = get_supports(train_baskets, eps)
    print(f"epsilon={eps}, zbiorów w słowniku: {len(s)}\n")
    for k in K:
        results = evaluate(test_baskets, products, s, k)
        print(f"{"mrr@" + str(k):<6} | {results[0].mrr:>6.4f} (vs {results[1].mrr:.4f})")
    print("\n")

epsilon=0.001, zbiorów w słowniku: 759

mrr@3  | 0.0383 (vs 0.0042)
mrr@5  | 0.0421 (vs 0.0167)
mrr@10 | 0.0343 (vs 0.0179)


epsilon=0.0005, zbiorów w słowniku: 1589

mrr@3  | 0.0550 (vs 0.0042)
mrr@5  | 0.0508 (vs 0.0167)
mrr@10 | 0.0423 (vs 0.0179)


epsilon=0.0001, zbiorów w słowniku: 6796

mrr@3  | 0.0458 (vs 0.0042)
mrr@5  | 0.0482 (vs 0.0167)
mrr@10 | 0.0488 (vs 0.0179)


epsilon=5e-05, zbiorów w słowniku: 51496

mrr@3  | 0.0350 (vs 0.0042)
mrr@5  | 0.0382 (vs 0.0167)
mrr@10 | 0.0412 (vs 0.0179)


epsilon=0, zbiorów w słowniku: 51496

mrr@3  | 0.0350 (vs 0.0042)
mrr@5  | 0.0382 (vs 0.0167)
mrr@10 | 0.0412 (vs 0.0179)


